# Embedding & Vector Indexing Pipeline

**Thesis:** RAG-Driven Natural Language Energy Demand Forecasting  
**Author:** Zoheb Anwar Hussain (Student ID: 1196931)  
**Embedding Model:** `all-MiniLM-L6-v2` (384-dim, local CPU, no API calls)

---

### What this notebook does
1. Loads the 480 KB summaries as LangChain `Document` objects via pandas
2. Embeds each summary using the local `all-MiniLM-L6-v2` sentence-transformer
3. Builds a **FAISS** index for dense similarity search (Pipeline 1)
4. Builds a **ChromaDB** collection with metadata filtering (Pipelines 2 & 3)
5. Validates both indexes with sample queries

### Architecture
All logic lives in `src/embedding/`. This notebook is thin orchestration.

### No API Calls
This entire phase runs locally on CPU. No Gemini, no Groq, no internet required.
The embedding model is downloaded from HuggingFace Hub on first run and cached locally.

---
## Cell 1 — Imports and Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from config import PATHS, EMBEDDING_MODEL_NAME
from config.paths import create_all_directories
from src.utils import setup_logger, log_section
from src.embedding import (
    load_kb_documents,
    get_embeddings_model,
    build_faiss_index,
    load_faiss_index,
    build_chroma_index,
    load_chroma_index,
)

logger = setup_logger("embedding")
create_all_directories()

logger.info("All imports successful.")
logger.info("Embedding model: %s", EMBEDDING_MODEL_NAME)
logger.info("FAISS save path: %s", PATHS['faiss_index'])
logger.info("ChromaDB persist path: %s", PATHS['chroma_index'])

---
## Cell 2 — Load KB Summaries as LangChain Documents

Each row of `combined_master_summaries.csv` becomes a `Document` with:
- `page_content` = summary text only (clean, no metadata noise)
- `metadata` = `{source, kb_id, row_id, dataset, granularity, zone_id, year, season, parent_id}`

In [ ]:
log_section("Loading KB summaries as Documents", 1, 5)

kb_csv_path = (
    PATHS["summaries_csv"] / "combined_master_summaries.csv"
)
documents = load_kb_documents(kb_csv_path)

logger.info("Loaded %d documents.", len(documents))

# Preview first document
print("\n--- Sample Document ---")
print(f"page_content (first 200 chars):")
print(f"  {documents[0].page_content[:200]}...")
print(f"\nmetadata:")
for k, v in documents[0].metadata.items():
    print(f"  {k:<15} = {v}")

---
## Cell 3 — Initialise Embedding Model

Downloads `all-MiniLM-L6-v2` from HuggingFace Hub on first run (~80 MB).
Cached locally at `~/.cache/huggingface/` — subsequent runs load from disk instantly.

In [ ]:
log_section("Initialising embedding model", 2, 5)

embeddings = get_embeddings_model()

# Quick dimension check
test_vector = embeddings.embed_query("peak electricity demand in winter")
logger.info(
    "Embedding dimension: %d. Normalised: %.4f (should be ~1.0).",
    len(test_vector),
    sum(v**2 for v in test_vector) ** 0.5,
)

---
## Cell 4 — Build FAISS Index

IndexFlatIP (exact inner product search). With normalised embeddings,
inner product equals cosine similarity.

Used by: **Pipeline 1 (dense)** and the dense component of **Pipeline 2 (hybrid)**.

In [ ]:
log_section("Building FAISS index", 3, 5)

faiss_vs = build_faiss_index(
    documents=documents,
    embeddings=embeddings,
    save_path=PATHS["faiss_index"],
)

logger.info("FAISS index ready.")

---
## Cell 5 — Build ChromaDB Collection

Persisted SQLite-backed collection with metadata filtering.
Stores all metadata fields for filtered retrieval.

Used by: **Pipeline 2 (hybrid, metadata filter)** and **Pipeline 3 (hierarchical, parent_id lookup)**.

In [ ]:
log_section("Building ChromaDB collection", 4, 5)

chroma_vs = build_chroma_index(
    documents=documents,
    embeddings=embeddings,
    persist_dir=PATHS["chroma_index"],
)

logger.info("ChromaDB collection ready.")

---
## Cell 6 — Validation Queries

Run test queries against both indexes to verify they return relevant results.

In [ ]:
log_section("Running validation queries", 5, 5)

test_queries = [
    "What is the peak electricity load in winter for Zone 4?",
    "How does kitchen appliance usage vary on weekdays?",
    "Compare summer and winter demand patterns across all zones",
    "What is the annual trend in household power consumption?",
]

print("\n" + "=" * 65)
print("  FAISS VALIDATION (dense similarity search)")
print("=" * 65)

for query in test_queries:
    results = faiss_vs.similarity_search_with_score(query, k=3)
    print(f"\n  Query: {query}")
    for doc, score in results:
        print(
            f"    [{score:.4f}] {doc.metadata.get('source', 'N/A'):<45} "
            f"({doc.metadata.get('dataset', '')}/{doc.metadata.get('granularity', '')})"
        )
        print(f"             {doc.page_content[:100]}...")

print("\n" + "=" * 65)
print("  CHROMADB VALIDATION (metadata-filtered search)")
print("=" * 65)

# Test 1: Filter by dataset + season
query = "What is the peak electricity load in winter?"
results = chroma_vs.similarity_search(
    query, k=3, filter={"dataset": "gefcom"}
)
print(f"\n  Query: {query}")
print(f"  Filter: dataset=gefcom")
for doc in results:
    print(
        f"    {doc.metadata.get('source', 'N/A'):<45} "
        f"({doc.metadata.get('dataset', '')}/{doc.metadata.get('granularity', '')})"
    )

# Test 2: Filter by granularity
query = "Weekly household consumption pattern"
results = chroma_vs.similarity_search(
    query, k=3, filter={"granularity": "weekly"}
)
print(f"\n  Query: {query}")
print(f"  Filter: granularity=weekly")
for doc in results:
    print(
        f"    {doc.metadata.get('source', 'N/A'):<45} "
        f"({doc.metadata.get('dataset', '')}/{doc.metadata.get('granularity', '')})"
    )

# Test 3: Parent_id lookup (hierarchical retrieval preview)
print("\n  Parent_id lookup test (hierarchical retrieval preview):")
sample_doc = documents[0]
parent_id  = sample_doc.metadata.get("parent_id", "")
print(f"    Child doc  : {sample_doc.metadata.get('source', 'N/A')}")
print(f"    parent_id  : {parent_id}")
if parent_id:
    parent_results = chroma_vs.similarity_search(
        "", k=1, filter={"row_id": parent_id}
    )
    if parent_results:
        print(
            f"    Parent doc : {parent_results[0].metadata.get('source', 'N/A')}"
        )
        print(f"    Parent text: {parent_results[0].page_content[:100]}...")
    else:
        print("    Parent not found in ChromaDB.")
else:
    print("    No parent_id for this document (expected for non-daily/weekly types).")

print("\n" + "=" * 65)

---
## Pipeline Complete ✅

Vector indexes saved to:
```
outputs/indexes/faiss/       ← FAISS binary index + metadata
outputs/indexes/chromadb/    ← ChromaDB persistent SQLite collection
```

Both indexes are loaded at the start of Phase 4 using:
```python
from src.embedding import load_faiss_index, load_chroma_index, get_embeddings_model

embeddings = get_embeddings_model()
faiss_vs   = load_faiss_index(PATHS['faiss_index'], embeddings)
chroma_vs  = load_chroma_index(PATHS['chroma_index'], embeddings)
```

### Next Phase
Move to `notebooks/04_retrieval_pipelines.ipynb` to implement the three
retrieval strategies: Dense (FAISS), Hybrid (BM25 + FAISS), Hierarchical (ChromaDB + parent_id).